In [7]:
import pandas as pd

df = pd.read_csv("data/mimic_train_impressions.csv")
impressions = df["report"].drop_duplicates().dropna().reset_index(drop = True)
print(len(impressions.to_list()))
print(len(df), len(impressions))


print(impressions.head())

130700
371951 130700
0                    No acute cardiopulmonary process.
1                No acute cardiopulmonary abnormality.
2                      No acute intrathoracic process.
3    Focal consolidation at the left lung base, pos...
4        No evidence of acute cardiopulmonary process.
Name: report, dtype: object


In [ ]:
from models.model_retrieval import ALBEF
import yaml 
from models.tokenization_bert import BertTokenizer
import torch
import torch.nn.functional as F

device = 'cuda' if torch.cuda.is_available() else 'cpu'

config = yaml.safe_load(open('configs/Pretrain.yaml'))

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = ALBEF(config=config, 
                                text_encoder='bert-base-uncased', 
                                tokenizer=tokenizer
                                ).to(device = device)
# load chkpt
ckpt = torch.load('chkpts/checkpoint_59.pth', map_location='cpu')
msg = model.load_state_dict(ckpt['model'], strict=False)
print(msg)                                   # check what's missing/unexpected
model = model.to(device).eval()              # eval mode = inference
####################################
@torch.no_grad()
def embed_texts(texts):
    tok = tokenizer(texts, padding='max_length', truncation=True,
                    max_length=40, return_tensors='pt').to(device)
    out = model.text_encoder(tok.input_ids, attention_mask=tok.attention_mask,
                             mode='text', return_dict=True)
    return F.normalize(model.text_proj(out.last_hidden_state[:, 0, :]), dim=-1).cpu().tolist()

In [ ]:
import chromadb, torch

client = chromadb.PersistentClient(path="./cxr_db")

collection = client.get_or_create_collection(
    name="impressions",
    metadata={"hnsw:space": "cosine"},
)